In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys
import os
from sklearn import set_config

from sksurv.linear_model import CoxnetSurvivalAnalysis
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from typing import Tuple
set_config(display="text")  # displays text representation of estimators

from sksurv.util import Surv
sys.path.append(os.path.abspath("../../"))
from src.utils.ConvertTextToCsv import TextToCsv
from src.utils.Preprocessing import Preprocessor
from src.utils.cox_models import Cox_regression, p_values_Cox_regression, plot_coefficients
from src.utils.plots import Plots
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
pp = Preprocessor()
cox_lasso = CoxnetSurvivalAnalysis(l1_ratio=1.0, alpha_min_ratio=0.01)
df_clinical_data = pd.read_csv("../../data/raw/brca_tcga_pub2015_clinical_data.tsv", sep='\t')
df_clinical_keep = pp.clean_columns_dataset(df_clinical_data)



In [3]:
list_df = pp.total_type_len_type_cancer(df_clinical_keep)
df_clinical_keep["Tumor-Cancer"] = list_df
df_mRNA_transformed = TextToCsv("../../data/raw/data_mrna_seq_v2_rsem.txt")
df_rsem_z_scores = TextToCsv("../../data/raw/data_mrna_seq_v2_rsem_zscores_ref_all_samples.txt")
clean_df = pp.eliminate_nan_genes(df_rsem_z_scores, "Hugo_Symbol")



Luminal A: 330 - Total(%): 0.40
Luminal B: 81 - Total(%):0.10
HER2-enriched: 23 - Total(%):0.03
TNBC: 85 - Total(%)0.10 
UNK: 298 - Total(%) 0.36
Shape of the CSV: (20440, 819)
Shape of the CSV: (20440, 819)
Max NaN per gene: 817
Genes before: 20440
Genes after: 20107


In [4]:

"""expression = df_ESR1_merged["expression"].values
lsit = sorted(expression)
plt.plot((lsit)) # plotting by columns
plt.show()
"""

'expression = df_ESR1_merged["expression"].values\nlsit = sorted(expression)\nplt.plot((lsit)) # plotting by columns\nplt.show()\n'

In [5]:

"""
ESR1 = df_ESR1_merged["expression"].values.reshape(-1,1)

scaler = StandardScaler()
z_score = scaler.fit_transform(ESR1)

plt.figure(figsize=(12,5))
plt.title("box plot of the z score")
plt.boxplot(z_score)
plt.show()

plt.figure(figsize=(12,5))
plt.hist(z_score, bins=30)
plt.title("Distribution of the z score")
plt.xlabel("z-score")
plt.ylabel("Frecuencia")
plt.show()
print(f"Min number ESR1: {ESR1.min()}")
print(f"Max number ESR1: {ESR1.max()}")
#plt.xlabel(f"Max number in ESR1:{ESR1.max()} ")"""

'\nESR1 = df_ESR1_merged["expression"].values.reshape(-1,1)\n\nscaler = StandardScaler()\nz_score = scaler.fit_transform(ESR1)\n\nplt.figure(figsize=(12,5))\nplt.title("box plot of the z score")\nplt.boxplot(z_score)\nplt.show()\n\nplt.figure(figsize=(12,5))\nplt.hist(z_score, bins=30)\nplt.title("Distribution of the z score")\nplt.xlabel("z-score")\nplt.ylabel("Frecuencia")\nplt.show()\nprint(f"Min number ESR1: {ESR1.min()}")\nprint(f"Max number ESR1: {ESR1.max()}")\n#plt.xlabel(f"Max number in ESR1:{ESR1.max()} ")'

In [4]:
def split_data_set(df_mRNA : pd.DataFrame, 
                       df_clinical : pd.DataFrame,
                       gene : str) -> Tuple:
    
        df_single_gene  = pp.gene_to_long(df_mRNA, gene)
        df_gene_merged = df_single_gene.merge(df_clinical, on="Sample ID", how="inner")
        
        status = df_gene_merged["Overall Survival Status"].astype(str).str.strip()
        
        df_gene_merged["event"] = status.str.contains("DECEASED", na=False) 
    
        df_gene_merged = df_gene_merged.dropna(subset=["Overall Survival (Months)"])
        
        df_gene_merged["expression"] = pd.to_numeric(df_gene_merged["expression"], errors="coerce")
        
        X = df_gene_merged[["expression"]]
        
        Y_surv = Surv.from_dataframe(
        event="event",
        time="Overall Survival (Months)",
        data=df_gene_merged
        )
        
        return X, Y_surv, df_gene_merged

In [5]:

X_ESR1, Y_surv_ESR1, df_gene_merged_ESR1 = split_data_set(clean_df,df_clinical_keep, "ESR1")
X_train_ESR1, X_test_ESR1, Y_train_ESR1, Y_test_ESR1 = train_test_split(
    X_ESR1, Y_surv_ESR1, train_size=0.80, test_size=0.20, random_state=42
)
betas_ESR1, chp_predict_ESR1, survival_curve_ESR1, risk_curve_ESR1 = Cox_regression(X_train_ESR1, Y_train_ESR1, X_test_ESR1)

In [6]:
df_life_ESR1 = df_gene_merged_ESR1[["expression", "event", "Overall Survival (Months)"]]
df_life_ESR1 = df_life_ESR1.dropna(subset="Overall Survival (Months)")
df_life_ESR1 = df_life_ESR1.dropna(subset="expression")
p_values_Cox_regression(df_life_ESR1,event_col="event", duration_col="Overall Survival (Months)")

,coef,exp(coef),se(coef),coef lower 95%,coef upper 95%,exp(coef) lower 95%,exp(coef) upper 95%,cmp to,z,p,-log2(p)
covariate,,,,,,,,,,,
expression,-0.005539,0.994477,0.097545,-0.196724,0.185646,0.821418,1.203996,0.0,-0.056782,0.954719,0.066852


In [11]:

X_TP53, Y_surv_TP53, df_gene_merged_TP53 = split_data_set(clean_df,df_clinical_keep, "TP53")
X_train_TP53, X_test_TP53, Y_train_TP53, Y_test_TP53 = train_test_split(
    X_TP53, Y_surv_TP53, train_size=0.80, test_size=0.20, random_state=42
)
betas_TP53, chp_predict_TP53, survival_curve_TP53, risk_curve_TP53 = Cox_regression(X_train_TP53, Y_train_TP53, X_test_TP53)

In [12]:
df_life_TP53 = df_gene_merged_TP53[["expression", "event", "Overall Survival (Months)"]]

df_life_TP53 = df_life_TP53.dropna(subset="Overall Survival (Months)")
df_life_TP53 = df_life_TP53.dropna(subset="expression")
p_values_Cox_regression(df_life_TP53,event_col="event", duration_col="Overall Survival (Months)")

,coef,exp(coef),se(coef),coef lower 95%,coef upper 95%,exp(coef) lower 95%,exp(coef) upper 95%,cmp to,z,p,-log2(p)
covariate,,,,,,,,,,,
expression,0.149478,1.161228,0.094539,-0.035814,0.334771,0.96482,1.39762,0.0,1.581134,0.113847,3.134826


In [13]:

X_MKI67, Y_surv_MKI67, df_gene_merged_MKI67 = split_data_set(clean_df,df_clinical_keep, "MKI67")
X_train_MKI67, X_test_MKI67, Y_train_MKI67, Y_test_MKI67 = train_test_split(
    X_MKI67, Y_surv_MKI67, train_size=0.80, test_size=0.20, random_state=42
)
betas_MKI67, chp_predict_MKI673, survival_curve_MKI67, risk_curve_MKI67 = Cox_regression(X_train_MKI67, Y_train_MKI67, X_test_MKI67)

df_life_MKI67 = df_gene_merged_MKI67[["expression", "event", "Overall Survival (Months)"]]
df_life_MKI67 = df_life_MKI67.dropna(subset="Overall Survival (Months)")
df_life_MKI67 = df_life_MKI67.dropna(subset="expression")
p_values_Cox_regression(df_life_MKI67,event_col="event", duration_col="Overall Survival (Months)")



,coef,exp(coef),se(coef),coef lower 95%,coef upper 95%,exp(coef) lower 95%,exp(coef) upper 95%,cmp to,z,p,-log2(p)
covariate,,,,,,,,,,,
expression,0.085822,1.089613,0.089302,-0.089206,0.260851,0.914657,1.298034,0.0,0.961037,0.336534,1.571178


In [14]:

X_BCL2, Y_surv_BCL2, df_gene_merged_BCL2 = split_data_set(clean_df,df_clinical_keep, "BCL2")
X_train_BCL2, X_test_BCL2, Y_train_BCL2, Y_test_BCL2 = train_test_split(
    X_BCL2, Y_surv_BCL2, train_size=0.80, test_size=0.20
)
beta_BCL2, chp_predict_BCL2, survival_curv_BCL2, risk_curve_BCL27 = Cox_regression(X_train_BCL2, Y_train_BCL2, X_test_BCL2)

df_life_BCL2 = df_gene_merged_BCL2[["expression", "event", "Overall Survival (Months)"]]
df_life_BCL2 = df_life_BCL2.dropna(subset="Overall Survival (Months)")
df_life_BCL2 = df_life_BCL2.dropna(subset="expression")
p_values_Cox_regression(df_life_BCL2,event_col="event", duration_col="Overall Survival (Months)")

,coef,exp(coef),se(coef),coef lower 95%,coef upper 95%,exp(coef) lower 95%,exp(coef) upper 95%,cmp to,z,p,-log2(p)
covariate,,,,,,,,,,,
expression,-0.225505,0.798113,0.094709,-0.411131,-0.039879,0.6629,0.960906,0.0,-2.381033,0.017264,5.856075
